# RunPod: V5, the one-stage Sieve

**The model.** V4's Sieve (B1's encoder (ViT-tiny, CLS, projector), predictor and action embedding, history 1, the prediction target detached, **no SIGReg, no IDM**, a pixel decoder that reads only the latent) trained in **one stage from the start** (`sieve/losses.py`, mode `joint`, config `craftax_sieve_joint`). One loss:
- **decoded prediction:** the predicted latent decoded and compared with the real next frame;
- **gated reconstruction of the next frame:** each patch weighted by `floor + (1 - floor) exp(-g / (beta * mean g))`, where `g` is the decoded prediction against the decoded latent of the next frame (`diff`: no pixels, so a patch the decoder cannot draw sharply is not shut where the prediction is right) and the mean is the batch's (no tau);
- **the next-latent MSE** against the stop-gradient target.

Early the errors are all alike and the gate is about uniform (0.43 at floor 0.1 and beta 1): every next frame is reconstructed, as in V4's stage 1. As the predictor gets content right, that content's error falls under the mean and its patches open (up to 1); what it cannot foresee falls towards the floor, as in V4's stage 2, but never to zero. No fork: V4's switch from stage 1 to stage 2, and the choice of its epoch, are gone.

**Why** (V4's first runs, 2026-11-17): stage 2's gate shut every rare thing (coal, iron, furnace, mobs) because the decoder could not draw it within tau even from the latent of the real frame (`sieve/gate_analysis.py`: `own_ok` about `sharp`), and content that is not available at the fork does not come back. The relative gate needs no tau, the `diff` error leaves the decoder's precision out, and the floor keeps every patch reconstructed a little.

**How this notebook runs.** One training as two workers on the one GPU, started detached (a pod restart: run the start cell again, the training resumes from `lightning_resume/last.ckpt` and the metrics skip the epochs done):
1. the training (`main_hjepa.py --config-name craftax_sieve_joint`), which saves the model at the start (epoch 0) and after every epoch;
2. the metrics worker (`python -m sieve.watch`), which reads each epoch checkpoint as it appears and rewrites the report (`<run>/sieve/report.md`).

The learning rate is constant after the warmup, so the run is **extended by resuming**: raise `EPOCHS` and run the start cell again.

**The metrics** are V4's (`sieve/check.py`, the fresh decoder included). The gate's own numbers are in the training log and the loss curves (panel "Sieve: gate open, sharp patches"): `gate_open_loss_level1` (the mean gate), `gate_std_loss_level1` (its spread: 0 is uniform, it should grow as the predictor learns) and `gate_shut_share_loss_level1` (patches near the floor).

**When to stop** (`sieve/report.py`, mode `joint`; heuristics): at two epochs in a row no headline probe rises, the noise moves the latent little and steadily, the rank moved by at most 5% and the train/held-out gaps did not grow. The report proposes that epoch.

**Against V4** (`data/runs/V4_seed42/`, stage-1 epoch 8 and stage-2 epoch 4): does V5 keep the mobs, coal, furnace and iron that V4's stage 2 lost, and gain the faced tile it gained? Section 3 runs the gate analysis on V5's last epoch, with the one-stage gate as a column.

Once per pod, in a terminal: `git clone https://github.com/quastAI/AAC.git /root/AAC` (the clone must contain this notebook, `sieve/` and `craftax_sieve_joint.yaml`: push them first), then open this notebook from the clone.

In [ ]:
# --- 0. Configuration ------------------------------------------------------------------------------------------------
import os, shutil, sys, time
from pathlib import Path

EXPERIMENT = "V5"                           # the one-stage Sieve: trains into V5/

REPO = Path("/root/AAC")                    # the git clone, on the local disk (pod.py has the other paths and why they are where they are)
sys.path.insert(0, str(REPO / "notebooks"))
import pod

HF_REPO_ID = "quastAI/craftax-classic-pqn-rnn-v1"   # the dataset repo of runpod_dataset.ipynb (data/), runs go to runs/
HF_PRIVATE = True
SEEDS = "42"                                # one seed
USE_WANDB = True                            # live curves in Weights & Biases (project hjepa-flat); False = the metrics CSV only

BATCH_SIZE = 256                            # V4's (its smoke run: 1723 clips/s at 256 against 1186 at 128, about 5 GiB of GPU memory)
LEARNING_RATE = 7e-5                        # constant after a 500-step warmup, V4's
EPOCHS = 10                                 # to extend: raise it and run the start cell again (it resumes)
GATE_FLOOR = 0.1                            # the lowest gate weight of a patch (heuristic)
GATE_BETA = 1.0                             # selectivity: at the batch's mean error the gate is GATE_FLOOR + (1 - GATE_FLOOR) e^(-1/GATE_BETA)
SMOKE_BATCHES = 300                         # training batches of the smoke run (the rate is measured after the first 100)

In [ ]:
# --- 0. Paths, Hugging Face login, the PyTorch environment (installed again on a new pod) and the dataset ----------------
pod.configure(repo=REPO, seeds=SEEDS)
from pod import DATA, H_JEPA, LOGS, PYTHON, launch, run_dir, running, sh, show

TRAIN_OVERRIDES = pod.train_overrides(USE_WANDB)   # Hydra overrides for every training run
api = pod.login(HF_REPO_ID, HF_PRIVATE, USE_WANDB)
pod.install(api, HF_REPO_ID, USE_WANDB)
SPEC = pod.run_spec(EXPERIMENT, EXPERIMENT)        # the training config (craftax_sieve_joint) and its overrides
FOLDER = pod.folder_of(EXPERIMENT)                 # the run trains into it
RUN = run_dir(FOLDER)
SEED = SEEDS.split(",")[0]
TRAIN, VAL = DATA / "craftax_expert_train.h5", DATA / "craftax_expert_val.h5"
BANK = DATA / "sieve_counterfactuals_val.npz"      # the re-rendered frames of the moves_for_* metrics (built below, once per pod; V4's file)
TRAIN_JOB, METRICS_JOB = EXPERIMENT, f"{EXPERIMENT}_metrics"
COMMON = [*SPEC.get("overrides", []), f"seed={SEED}", f"level1.lr={LEARNING_RATE}", f"optimizer.lr={LEARNING_RATE}",
          f"level1.sieve.gate_floor={GATE_FLOOR}", f"level1.sieve.gate_beta={GATE_BETA}", *TRAIN_OVERRIDES]
print(EXPERIMENT, FOLDER, SPEC["config"], *SPEC.get("overrides", []))

## 1. The fixed inputs of the metrics, then the smoke run

The metrics read the same frames at every epoch: 24,576 frames of training worlds (the probes and the fresh decoder are fitted there) and 16,384 of held-out worlds, cached next to the data. The counterfactual bank re-renders stored simulator states after one edit (facing, a cow, the faced tile, an item) and night frames with another noise draw; it checks that an unedited render equals the stored frame (`render_match` 1.0). Both are V4's: on a pod that ran V4 they exist already.

The smoke run trains `SMOKE_BATCHES` batches at `BATCH_SIZE` and runs one metrics pass on epoch 0 and 1. It prints the steady steps per second, the GPU use and memory, the hours per epoch, the seconds per metrics check and the gradient ratio `recon_vs_pred_gradient`. The clip count is read with the venv's python (the kernel has no h5py).

In [ ]:
# --- 1. The sample of the metrics and the counterfactual bank (once per pod; CPU) --------------------------------------------
t0 = time.time()
if not BANK.exists():
    sh([PYTHON, "-m", "sieve.counterfactuals", "--val-file", VAL, "--out", BANK], cwd=H_JEPA, quiet=True, log="sieve_bank")
show(BANK.with_suffix(".json"), 30)
sh([PYTHON, "-c", "import sys; from sieve import sample; s = sample.load(sys.argv[1], sys.argv[2]); print(len(s.train), 'training and', len(s.val), 'held-out frames')",
    TRAIN, VAL], cwd=H_JEPA)
print(f"inputs ready in {time.time() - t0:.0f} s")

In [ ]:
# --- 1. Smoke run: SMOKE_BATCHES batches at BATCH_SIZE, one metrics pass of epoch 0 and 1 (full logs in LOGS/V5_smoke*.log) ---------
import json, subprocess
SMOKE = f"{EXPERIMENT}_smoke"
COUNT = "import sys, h5py; f = h5py.File(sys.argv[1], 'r'); print(int((f['ep_len'][:].astype(int) - 1).clip(min=0).sum()))"
clips = int(subprocess.run([str(PYTHON), "-c", COUNT, str(TRAIN)], env=pod.ENV, capture_output=True, text=True, check=True).stdout)
shutil.rmtree(run_dir(SMOKE), ignore_errors=True)
with pod.GpuSampler() as gpu:
    sh([PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *COMMON, "trainer.max_epochs=1", f"+trainer.limit_train_batches={SMOKE_BATCHES}",
        "+trainer.limit_val_batches=2", "resume_every_n_steps=0", "wandb.enabled=false",          # no W&B run for a smoke run
        f"output_model_name={SMOKE}", f"loader.batch_size={BATCH_SIZE}"], cwd=H_JEPA, quiet=True, log=SMOKE)
rate = pod.steady_rate(LOGS / f"{SMOKE}.log")
sh([PYTHON, "-m", "sieve.watch", "--run", run_dir(SMOKE), "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK, "--once"],
   cwd=H_JEPA, quiet=True, log=f"{SMOKE}_metrics")
m = [json.loads((run_dir(SMOKE) / "sieve" / f"epoch_{e}.json").read_text()) for e in (0, 1)]
hours = clips / BATCH_SIZE / rate / 3600 if rate else float("nan")
print(f"{clips:,} clips per epoch; batch {BATCH_SIZE}: {rate or float('nan'):.2f} steps/s, {(rate or 0) * BATCH_SIZE:.0f} clips/s, "
      f"GPU {gpu.utilization or float('nan'):.0f}%, {gpu.memory or float('nan'):.0f} MiB, {hours:.2f} h per epoch")
print(f"estimate: {EPOCHS} epochs {EPOCHS * hours:.1f} h (the training only; V4's metrics worker beside it added about 2 minutes per epoch)")
print(f"metrics check: {m[1]['seconds']:.0f} s per epoch (alone; next to the training it took about twice as long in V4)")
print(f"recon_vs_pred_gradient: start {m[0].get('recon_vs_pred_gradient', float('nan')):.2f}, after {SMOKE_BATCHES} steps "
      f"{m[1].get('recon_vs_pred_gradient', float('nan')):.2f}")

## 2. The run

Starts the training and the metrics worker, detached. To **extend**, raise `EPOCHS` and run the start cell again: the training resumes from its last state (constant rate) and the worker evaluates only the new epochs. To stop early after an epoch is saved: `os.killpg(int((LOGS / f"{TRAIN_JOB}.pid").read_text()), 15)`; the worker reads the remaining checkpoints and ends. The epoch checkpoints stay on the volume and are not uploaded: download the ones to keep.

In [ ]:
# --- 2. Start (or resume, or extend) the run: the training and the metrics worker -------------------------------------------
launch(TRAIN_JOB, [PYTHON, "main_hjepa.py", "--config-name", SPEC["config"], *COMMON, f"output_model_name={FOLDER}",
                   f"loader.batch_size={BATCH_SIZE}", f"trainer.max_epochs={EPOCHS}", f"wandb.config.name={EXPERIMENT}"], cwd=H_JEPA)
launch(METRICS_JOB, [PYTHON, "-m", "sieve.watch", "--run", RUN, "--train-file", TRAIN, "--val-file", VAL, "--bank", BANK,
                     "--until-epoch", EPOCHS, "--training-pid-file", LOGS / f"{TRAIN_JOB}.pid"], cwd=H_JEPA)

In [ ]:
# --- 2. Progress of both workers (interrupt any time: the jobs keep running; run it again to follow them) -----------------------
pod.follow(pod.sieve_bars(TRAIN_JOB, METRICS_JOB, RUN, EPOCHS))
print("\n".join(pod.progress(TRAIN_JOB, "[health", 2) + pod.progress(METRICS_JOB, "[sieve]", 3)))

In [ ]:
# --- 2. Loss curves (also while it trains; the gate's own numbers are in the gate panel) and the report: the epoch to stop at ------
import json
from IPython.display import Image, display
sh([PYTHON, REPO / "analysis" / "plot_training.py", "--run", RUN], check=False)
if (RUN / "health" / "loss_curves.png").exists():
    display(Image(filename=str(RUN / "health" / "loss_curves.png")))
show(RUN / "sieve" / "report.md", 200)
if (RUN / "sieve" / "report.png").exists():
    display(Image(filename=str(RUN / "sieve" / "report.png")))
proposal = json.loads((RUN / "sieve" / "report.json").read_text()).get("ready_epoch") if (RUN / "sieve" / "report.json").exists() else None
print("proposed stop epoch:", proposal if proposal is not None else "none yet (extend the run if the probes still rise)")

## 3. What the gate keeps (optional, a few minutes)

The gate analysis of V4 (`sieve/gate_analysis.py`) on V5's newest epoch checkpoint, 64 held-out episodes: per patch the decoded prediction, the decoder's own error (`own_ok`, the ceiling) and every gate side by side (the recipe's `abs`, `rel`, `diff` and the one-stage gate of this run), grouped by what the patch shows (block, mob, inventory strip; entering, changed or unchanged tile; daylight). The labels only group the patches. Compare with V4's `gate_analysis.md` files in `data/runs/V4_seed42/`.

In [ ]:
# --- 3. The gate analysis of the newest epoch checkpoint (the labels only group the patches) ------------------------------------------
newest = max((int(p.name.split("_epoch_")[1].split("_")[0]), p) for p in RUN.glob(f"{FOLDER}_epoch_*_object.ckpt"))
GATE_OUT = RUN / "sieve" / f"gate_epoch_{newest[0]}"
sh([PYTHON, "-m", "sieve.gate_analysis", "--parent", newest[1], "--val-file", VAL, "--val-episodes", 64,
    "--gate-floor", GATE_FLOOR, "--gate-beta", GATE_BETA, "--out", GATE_OUT], cwd=H_JEPA, quiet=True, log=f"{EXPERIMENT}_gate")
show(GATE_OUT / "gate_analysis.md", 400)

## 4. Upload

The run goes to `runs/V5/` of the Hub repo: the final model, `sieve/` with every epoch's metrics and the gate analyses, the health reports and the config. The epoch checkpoints and the resumable state stay on the volume. The report goes to `reports/` as `V5.md` and `V5.png`.

In [ ]:
# --- 4. The report and the run to the Hub -----------------------------------------------------------------------------------------------
for suffix in (".md", ".png"):
    if (RUN / "sieve" / f"report{suffix}").exists():   # copyfile: the volume refuses the file times copy2 sets
        shutil.copyfile(RUN / "sieve" / f"report{suffix}", DATA / "reports" / f"{EXPERIMENT}{suffix}")
pod.upload(api, HF_REPO_ID, folders=[FOLDER], reports=[f"{EXPERIMENT}.md", f"{EXPERIMENT}.png"])